In [0]:
from pyspark.sql import functions as F

# 1. Identify which orders have a "current" status in this batch
active_order_ids = (spark.table("workspace.amazon_fullfilment_silver.orders_silver")
    .filter("is_current = true")
    .select("order_id")
    .distinct()
)

# 2. Get the full history for ONLY those orders so we can find the "Initial" timestamp
# We join the full table with our list of active IDs
silver_updates = (spark.table("workspace.amazon_fullfilment_silver.orders_silver")
    .join(active_order_ids, "order_id", "inner") 
    .groupBy("order_id", "customer_id")
    .agg(
        F.min(F.when(F.col("status") == 'Initial', F.col("updated_timestamp"))).alias("initial_ts"),
        F.min(F.when(F.col("status") == 'Picking', F.col("updated_timestamp"))).alias("picking_ts"),
        F.min(F.when(F.col("status") == 'Shipped', F.col("updated_timestamp"))).alias("shipped_ts"),
        F.last(F.when(F.col("is_current") == True, F.col("status"))).alias("latest_status")
    )
    # 3. Use a Coalesce fallback for the date_key
    # If for some reason 'Initial' is missing, we use the next available timestamp
    .withColumn("valid_initial_ts", F.coalesce("initial_ts", "picking_ts", "shipped_ts"))
    .withColumn("date_key", F.date_format("valid_initial_ts", "yyyyMMdd").cast("int"))
    
    .withColumn("is_completed", F.when(F.col("latest_status") == 'Shipped', True).otherwise(False))
    
    .withColumn("total_fulfillment_time", 
        F.when(F.col("shipped_ts").isNotNull(), 
            (F.unix_timestamp("shipped_ts") - F.unix_timestamp("initial_ts")) / 60
        ).otherwise(None).cast("int")
    )
)

silver_updates.createOrReplaceTempView("updates_view1")

# 3. Perform the CUMULATIVE MERGE
spark.sql("""
MERGE INTO workspace.amazon_fullfilment_gold.fact_order_lifecycle AS target
USING updates_view1 AS source
ON target.order_id = source.order_id
WHEN MATCHED THEN
  UPDATE SET 
    target.date_picking = COALESCE(target.date_picking, source.picking_ts),
    target.date_shipped = COALESCE(target.date_shipped, source.shipped_ts),
    target.current_status = source.latest_status,
    target.is_completed = source.is_completed,
    target.total_fulfillment_time = source.total_fulfillment_time,
    -- Calculate internal milestone durations
    target.minutes_to_pick = CAST((unix_timestamp(source.picking_ts) - unix_timestamp(target.date_initial)) / 60 AS INT),
    target.minutes_to_ship = CAST((unix_timestamp(source.shipped_ts) - unix_timestamp(source.picking_ts)) / 60 AS INT),
    target._last_updated_timestamp = current_timestamp()
WHEN NOT MATCHED THEN
  INSERT (
    order_id, customer_id, date_key, 
    date_initial, date_picking, date_shipped, 
    total_fulfillment_time, current_status, is_completed, _last_updated_timestamp
  )
  VALUES (
    source.order_id, source.customer_id, source.date_key, 
    source.initial_ts, source.picking_ts, source.shipped_ts, 
    source.total_fulfillment_time, source.latest_status, source.is_completed, current_timestamp()
  )""")